# Version control, promotion and rollback for a Genie space
### dev → qa → prod with a complete, working CI/CD pipeline

You build a Genie space (shown as a **Genie Agent** in the Databricks UI) in a **dev** workspace by clicking around:
editing instructions, adding example queries, changing data sources. Out of the box those edits have **no version
history**: there is no "undo to last Tuesday", no record of who changed what, no safe way to move the space to QA and
production, and no protection against a mistake reaching business users.

This notebook builds all of that, and **runs it**:

| You get | How |
|---|---|
| **A version for every change made in dev**, including UI clicks | hourly snapshots of the dev space into a version history |
| **Undo** in any environment | `rollback` to the previous release, any snapshot or backup, or any git version |
| **Reviewed changes** | the space is exported to a file in git; every change is a pull request with a readable diff |
| **Promotion dev → qa → prod** | one environment-neutral file, deployed to each environment with its own catalog |
| **Quality gates** | static checks and SQL tests on every PR; Genie **benchmarks** in QA; a smoke test in prod |
| **Protection** | approval before prod; drift detection when someone edits QA/prod in the UI; a backup before every deploy; etag-guarded updates |
| **Audit trail** | who deployed which version, when, from which commit, to which environment |

Everything is implemented in this repository (`genie_cicd/`, `deploy/genie/`, `.github/workflows/genie-*.yml`) and
tested. Because this notebook can't reach your Databricks workspaces, Part 6 runs the **real code** against three
**simulated** workspaces (dev, qa, prod) on the local FreshCart data. Part 8 shows the exact steps to switch to
real Databricks workspaces: the code is the same, only `--target sim` becomes `--target databricks`.

## Contents

| Part | |
|---|---|
| 1 | The problem: what goes wrong without version control |
| 2 | The solution in one picture |
| 3 | Concepts you need (short) |
| 4 | The implementation: files, configuration, commands |
| 5 | The space file: what's in it, and how one file serves every environment |
| 6 | **Hands-on**: the full lifecycle, run step by step (simulated dev, qa, prod) |
| 7 | The CI/CD pipeline on GitHub Actions, job by job |
| 8 | Going live on Databricks: one-time setup, step by step |
| 9 | Alternative: Declarative Automation Bundles (`genie_spaces` resource) |
| 10 | Runbook: everyday tasks and emergencies |
| 11 | Summary and glossary |

**To run:** `pip install -r notebooks/requirements.txt`, open this notebook from the repository and *Run All*
(about 30 seconds; it builds the local warehouse if needed).

---
# 1 · The problem

| What happens | Without version control | With this setup |
|---|---|---|
| Someone rewrites the general instructions in dev and answers get worse | Nobody knows what the old text was | Every version is in dev's history; `rollback --env dev --to <version>` |
| An example query is "improved" and now uses gross instead of net sales | It's copied to prod by hand and executives see wrong numbers | The QA **benchmark gate** fails and prod promotion is blocked |
| Prod needs the change that was tested in QA | Re-click everything in prod's UI, slightly differently | The **same file** that passed QA is deployed to prod |
| Someone "quickly fixes" something directly in the prod UI | The next manual copy silently wipes it out, or it never reaches git | **Drift detection** refuses to overwrite it; the edit is shown and backed up |
| A release causes complaints on Monday morning | Try to remember what it looked like on Friday | `rollback --env prod --to previous`: one command, about a minute |
| The auditor asks who changed the prod assistant in March | No answer | History records: version, commit, who, when, what changed |
| QA uses `freshcart_qa.*` tables, prod uses `freshcart.*` | Hand-edit every table name and every SQL query | `{{catalog}}` placeholder, filled in per environment automatically |

---
# 2 · The solution in one picture

```
  DEVELOPERS                           GIT (GitHub)                              RELEASE PIPELINE (genie-release)
  ──────────                           ────────────                              ────────────────────────────────
  edit the DEV space in the UI
        │
        │ every hour: snapshot ───► DEV HISTORY (every UI change = a version, undo with rollback)
        │
        │ pull (export) ──────────► genie/space/freshcart_assistant.geniespace.json
        │                           (environment-neutral, canonical, readable diff)
        │                                    │
        │                           pull request ── genie-ci: validate · SQL runs · tests · change list
        │                                    │      + reviewer approval (branch protection)
        │                                    ▼
        │                           merge to main ─────────────────────────►  build: validate, package ONE artifact
        │                                                                            │
        │                                                                     QA:   deploy ─► benchmark gate ─► smoke
        │                                                                            │        (fails? stop here)
        │                                                                     [ approval by a reviewer ]
        │                                                                            │
        │                                                                     PROD: deploy ─► smoke ─► tag genie-prod-<version>
        │                                                                            │        (fails? automatic rollback)
        ▼                                                                            ▼
  every deploy, in every environment:  validate → dependencies exist → BACKUP live → drift guard → create/update (etag) → verify → RECORD
  rollback (genie-rollback, manual):   previous release │ any history record │ git:<tag or commit>
```

### Three layers of version history

| Layer | What it records | Where | Used for |
|---|---|---|---|
| **Git commits** (+ `genie-prod-*` tags) | every *reviewed* change to the space file | GitHub | code review, "what changed between releases", rollback to any git version |
| **Environment history** | `snapshot` (dev UI edits, hourly), `backup` (live state just before every deploy), `deployed` / `rollback` (every release), `gate` (benchmark results) | a Unity Catalog volume per environment, e.g. `/Volumes/freshcart/ops/genie_history/prod/` | undo in dev, emergency rollback, audit trail. Works even if git or CI is down |
| **The space itself** | the live state | the Genie space (same space id across releases) | users keep their conversations: deploys **update** the space in place, never recreate it |

---
# 3 · Concepts you need

> 💡 **Serialized space.** Databricks can export a whole Genie space (data sources, column settings, instructions,
> example SQL, SQL expressions, trusted functions, benchmarks) as one JSON document, and create or update a space from
> it. APIs: `GET /api/2.0/genie/spaces/{id}?include_serialized_space=true`, `POST /api/2.0/genie/spaces`,
> `PATCH /api/2.0/genie/spaces/{id}` (SDK: `w.genie.get_space / create_space / update_space`). Declarative Automation
> Bundles store the same JSON in a `.geniespace.json` file. **That JSON is what we version.**

> 💡 **Environment-neutral file.** Dev reads `freshcart_dev.semantic.sales_metrics`, prod reads
> `freshcart.semantic.sales_metrics`. In git, every such reference is written `{{catalog}}.semantic.sales_metrics`.
> **Render** fills in the environment's catalog just before a deploy; **neutralise** puts the placeholder back when a
> space is exported. One file, every environment, no hand edits.

> 💡 **Canonical form and content hash.** The file is always written the same way (sorted keys, sorted lists, one
> value per line), so git diffs show only real changes. A **content hash** (12 characters) identifies a version: two
> spaces with the same hash behave identically. It's how the tools know "has this changed?".

> 💡 **Artifact promotion: build once, deploy many.** The release pipeline packages the file once. QA gets that
> artifact; prod gets **the same artifact** after QA passed. Nothing is rebuilt or re-exported in between, so what was
> tested is exactly what ships.

> 💡 **Drift.** QA and prod are changed only by the pipeline. If the live space differs from the last release, someone
> edited it in the UI. That's *drift*. The pipeline refuses to silently overwrite it: it shows the change, keeps a
> backup, and lets you either adopt the change (pull it into git) or overwrite it on purpose.

> 💡 **Etag (optimistic concurrency).** Each read of a space returns an `etag` (a version stamp). The update sends it
> back; if the space changed in between, the API rejects the update instead of overwriting someone's edit made a
> second earlier.

> 💡 **Benchmarks as a quality gate.** Genie benchmarks are questions with a known correct SQL answer. Databricks can
> run them all against a space (`w.genie.genie_create_eval_run`) and grade each answer `GOOD`, `BAD` or `NEEDS_REVIEW`.
> The QA gate passes only if **nothing is BAD** and accuracy is above the threshold. Every trusted example query is
> also added as a benchmark, so "the examples still give the same answer" is checked on every release.

> 💡 **Rollback = redeploy an earlier version.** Because a deploy is a full replacement of the space's content, going
> back is simply deploying an older file through the same safe path (with a backup of the current state first, so the
> rollback can itself be undone).

---
# 4 · The implementation

| File | What it is |
|---|---|
| `genie/space/freshcart_assistant.geniespace.json` | **The space, as code.** Environment-neutral, canonical. The single source of truth that gets deployed |
| `deploy/genie/environments.yml` | What differs per environment: catalog, warehouse, title, folder, history location, gate thresholds |
| `genie_cicd/spec.py` | Load / save / normalise / neutralise / render / diff / validate the space file |
| `genie_cicd/workspace.py` | `DatabricksWorkspace` (real APIs via the Databricks SDK) and `SimulatedWorkspace` (local, same methods) |
| `genie_cicd/history.py` | The environment history: one JSON file per snapshot / backup / release / gate result |
| `genie_cicd/ops.py` | `pull`, `snapshot`, `plan`, `deploy`, `gate`, `smoke`, `promote`, `rollback` |
| `genie_cicd/cli.py` | `python -m genie_cicd <command>`, used by people and by the workflows |
| `genie_cicd/bootstrap.py` | One-time: compile the existing design in `genie/` into the first space file |
| `.github/workflows/genie-ci.yml` | Checks on every pull request |
| `.github/workflows/genie-release.yml` | main → QA → gate → approval → prod |
| `.github/workflows/genie-rollback.yml` | One-click rollback of any environment |
| `.github/workflows/genie-dev-sync.yml` | Hourly: snapshot dev, export it, open a sync pull request |
| `tests/test_genie_cicd.py` | Tests for all of the above, including the full lifecycle |

### Setup for this notebook

In [1]:
import os, sys, json, shutil, subprocess, tempfile, dataclasses, copy
from datetime import datetime, timedelta, timezone
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "genie_cicd").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))

import pandas as pd
from freshcart import config as FC, pipeline
from genie_cicd import ops, spec as S
from genie_cicd.history import HistoryStore
from genie_cicd.workspace import SimulatedWorkspace

pd.set_option("display.max_colwidth", 90)
if not (FC.WAREHOUSE_DIR / "gold.db").exists():          # the simulator runs SQL on the local FreshCart warehouse
    pipeline.run(full_refresh=True, verbose=False)

CONFIG = ops.load_config()
print("space file:", CONFIG.space_file.relative_to(ROOT))
print("environments:", " -> ".join(CONFIG.order))

space file: genie/space/freshcart_assistant.geniespace.json
environments: dev -> qa -> prod


### The environment configuration

In [2]:
print((ROOT / "deploy/genie/environments.yml").read_text())

# Genie space promotion: dev -> qa -> prod
# ---------------------------------------------------------------------------------------------
# One environment-neutral space file in git (catalog written as {{catalog}}) is deployed to every
# environment below. Only what differs between environments lives here.
#
# Values like ${GENIE_DEV_WAREHOUSE_ID} are read from environment variables at run time. In GitHub
# Actions they come from the variables of the matching GitHub environment (dev / qa / prod).
# On a laptop, set them in your shell, and authenticate with the `profile` from ~/.databrickscfg
# (the profile is used only when DATABRICKS_HOST is not set, so CI ignores it).

space:
  file: ../../genie/space/freshcart_assistant.geniespace.json
  title: FreshCart Sales & Stock Assistant
  description: |
    Ask about sales, margin, baskets, promotions, loyalty and stock across FreshCart stores
    and online. Figures are USD, exclude tax and use the FreshCart fiscal calendar.
    You only s

Note the differences: **catalog** (each environment reads its own data), **title** (`[DEV]`, `[QA]` suffixes so nobody
confuses them), **protected** (QA and prod may only be changed by the pipeline), **gate** (QA must pass the
benchmarks before anything is promoted to prod), and **history_root** (each environment keeps its history on a Unity
Catalog volume in its own catalog).

### The commands

In [3]:
print(subprocess.run([sys.executable, "-m", "genie_cicd", "--help"], capture_output=True, text=True).stdout)

usage: python -m genie_cicd [-h] [--config CONFIG] [--target {databricks,sim}]
                            [--sim-dir SIM_DIR]
                            {validate,fmt,diff,bootstrap,render,pull,snapshot,plan,deploy,gate,smoke,promote,rollback,history}
                            ...

Command line for Genie space version control and promotion.

    python -m genie_cicd validate [--sql]                 static checks (+ run every SQL on the local warehouse)
    python -m genie_cicd bootstrap                        compile genie/ design files into the space file (once)
    python -m genie_cicd diff     --base OLD.json         readable change list (used in pull requests)
    python -m genie_cicd render   --env qa [--out F]      the space file with the env's catalog filled in
    python -m genie_cicd pull     --env dev               live space -> space file in git (then open a PR)
    python -m genie_cicd snapshot --env dev               record the live space in history if it changed
    p

---
# 5 · The space file

The file was created once from the design that already existed in `genie/` (`python -m genie_cicd bootstrap`).
From now on **this file is what gets deployed**. Here is what it contains:

In [4]:
space = S.load(CONFIG.space_file)
print("content hash:", S.content_hash(space))
pd.DataFrame(S.summary(space).items(), columns=["section", "items"])

content hash: 8c1275b150c0


,section,items
0,table,2
1,metric view,2
2,sample question,4
3,text instruction,1
4,example SQL,6
5,trusted function,1
6,SQL filter,2
7,SQL expression,1
8,benchmark,17


| Section in the JSON | What it is in the Genie UI |
|---|---|
| `data_sources.tables`, `data_sources.metric_views` | Configure → Data: the tables and metric views Genie may query, with column settings (hidden columns, entity matching) |
| `instructions.text_instructions` | Configure → Instructions → Text: the general instructions |
| `instructions.example_question_sqls` | Example SQL queries (trusted answers), with parameters |
| `instructions.sql_snippets` | SQL expressions: filters, dimensions, measures |
| `instructions.sql_functions` | Trusted functions (here: like-for-like sales) |
| `config.sample_questions` | The suggested questions users see |
| `benchmarks.questions` | Benchmark questions with their correct SQL |

One example query, as stored. SQL is a **list of lines**, so a git diff shows exactly which line changed:

In [5]:
example = next(e for e in space["instructions"]["example_question_sqls"] if "net sales and margin by region" in S.sql_text(e["question"]))
print(json.dumps(example, indent=2))

{
  "id": "08131930aafcd3a8773021c860fd5d37",
  "question": [
    "What were net sales and margin by region last week?"
  ],
  "sql": [
    "SELECT region,\n",
    "       MEASURE(net_sales)        AS net_sales,\n",
    "       MEASURE(gross_margin)     AS gross_margin,\n",
    "       MEASURE(gross_margin_pct) AS gross_margin_pct\n",
    "FROM   {{catalog}}.semantic.sales_metrics\n",
    "WHERE  is_last_completed_fiscal_week\n",
    "GROUP BY ALL\n",
    "ORDER BY net_sales DESC"
  ]
}


### One file, every environment
`render` fills in each environment's catalog; `neutralise` reverses it. The round trip gives back the identical file:

In [6]:
for name in CONFIG.order:
    env = CONFIG.env(name)
    rendered = S.render(space, env.catalog)
    first_line = S.sql_text(next(e for e in rendered["instructions"]["example_question_sqls"] if e["id"] == example["id"])["sql"]).splitlines()[4]
    back = S.neutralise(rendered, env.catalog)
    print(f"{name:<5} catalog {env.catalog:<14} SQL says: {first_line.strip():<45} round trip identical: {S.content_hash(back) == S.content_hash(space)}")

dev   catalog freshcart_dev  SQL says: FROM   freshcart_dev.semantic.sales_metrics   round trip identical: True
qa    catalog freshcart_qa   SQL says: FROM   freshcart_qa.semantic.sales_metrics    round trip identical: True
prod  catalog freshcart      SQL says: FROM   freshcart.semantic.sales_metrics       round trip identical: True


### Static validation (what `genie-ci` runs on every pull request)
Checks that need no workspace: valid structure, unique ids, **no hard-coded environment catalog**, every SQL refers
only to objects the space has access to, benchmarks have SQL answers, plus (`--sql`) **every example and benchmark
query actually runs** on the local warehouse.

In [7]:
errors, warnings = S.validate(space, forbidden_catalogs=CONFIG.catalogs)
print("errors:", errors or "none", "| warnings:", warnings or "none")
print("SQL failures on the local warehouse:", ops.check_sql_locally(space) or "none")

# what a common mistake looks like: someone pastes a dev table name into the file
oops = copy.deepcopy(space)
oops["data_sources"]["tables"][0]["identifier"] = "freshcart_dev.gold.dim_store"
print("\nwith a hard-coded dev catalog ->", S.validate(oops, forbidden_catalogs=CONFIG.catalogs)[0])

errors: none | warnings: none


SQL failures on the local warehouse: none

with a hard-coded dev catalog -> ["hard-coded catalog 'freshcart_dev.' found; use '{{catalog}}.' so the file works in every environment"]


---
# 6 · Hands-on: the full lifecycle

We now play out a realistic month in the life of the FreshCart assistant, using the real `genie_cicd` code against
**three simulated workspaces**. The simulator keeps each workspace's spaces and files in a folder and runs SQL on the
local warehouse. It is not an AI: asked a question, it answers with the space's trusted example SQL for that exact
question, or reports `NEEDS_REVIEW`, as Genie benchmarks do for answers they can't grade automatically. On Databricks
the same calls go to the real Genie APIs.

We also use a **scratch git repository** (a temporary copy) so we can commit, tag and roll back freely without
touching this repository.

### 6.1 · Set up the lab

In [8]:
LAB = Path(tempfile.mkdtemp(prefix="genie_lab_"))
REPO = LAB / "repo"
SPACE_FILE = REPO / "genie/space/freshcart_assistant.geniespace.json"
SPACE_FILE.parent.mkdir(parents=True)
shutil.copy(CONFIG.space_file, SPACE_FILE)
cfg = dataclasses.replace(CONFIG, space_file=SPACE_FILE)       # same environments, space file in the scratch repo

def git(*args, quiet=False):
    # run git in the scratch repository; print the output, or return it with quiet=True
    out = subprocess.run(["git", "-C", str(REPO), *args], capture_output=True, text=True, check=True).stdout.strip()
    if quiet:
        return out
    if out:
        print(out)

def add_line(content, text):
    # append a line to an instruction the way the UI stores it (a list of lines)
    if content and not content[-1].endswith("\n"):
        content[-1] += "\n"
    content.append(text)

git("init", "-q", "-b", "main"); git("config", "user.name", "Dana Developer"); git("config", "user.email", "dana@freshcart.example")

class Clock:                                                    # a fixed, advancing clock so record ids are reproducible
    def __init__(self): self.t = datetime(2026, 10, 1, 9, 0, tzinfo=timezone.utc)
    def __call__(self): self.t += timedelta(minutes=7); return self.t
clock = Clock()

ENV = {}
for name in cfg.order:
    env = cfg.env(name)
    ws = SimulatedWorkspace(LAB / "workspaces" / name, env.catalog)
    ENV[name] = (env, ws, HistoryStore(ws, env.history_root, clock=clock))
(dev, dws, dh), (qa, qws, qh), (prod, pws, ph) = ENV["dev"], ENV["qa"], ENV["prod"]

def history(name):
    rows = ENV[name][2].table(name)
    return pd.DataFrame(rows)[["record_id", "kind", "content_hash", "version", "actor", "note"]].fillna("") if rows else "(empty)"

print("lab:", LAB)
for name, (env, ws, h) in ENV.items():
    print(f"  {name:<5} {ws.name:<40} title {env.title!r:<44} history {env.history_root}")

lab: /tmp/genie_lab_zm0b9pk4
  dev   simulated workspace (freshcart_dev)      title 'FreshCart Sales & Stock Assistant [DEV]'    history /Volumes/freshcart_dev/ops/genie_history
  qa    simulated workspace (freshcart_qa)       title 'FreshCart Sales & Stock Assistant [QA]'     history /Volumes/freshcart_qa/ops/genie_history
  prod  simulated workspace (freshcart)          title 'FreshCart Sales & Stock Assistant'          history /Volumes/freshcart/ops/genie_history


### 6.2 · Version 1 goes into git
The space file is committed. From this moment every change has a history.

In [9]:
git("add", "-A"); git("commit", "-q", "-m", "Genie space v1: bootstrap from the reviewed design")
git("log", "--oneline")

a5666fd Genie space v1: bootstrap from the reviewed design


### 6.3 · First deploy to dev
`deploy` runs the same safety steps every time: **validate → check that every table, metric view and function
exists in the target catalog → back up the live space (none yet) → drift check → create or update (with the etag) →
re-read and verify → record the release in history.**

In [10]:
r = ops.deploy(dws, dev, S.load(SPACE_FILE), dh, config=cfg, version="v1", git_sha=git("rev-parse", "--short", "HEAD", quiet=True), actor="dana")
live = dws.get_space(r["space_id"])
print("\nlive dev space:", {k: live[k] for k in ("space_id", "title", "parent_path", "etag")})
print("dev reads:", [t["identifier"] for t in live["serialized_space"]["data_sources"]["metric_views"]])
history("dev")

  created dev space bbe0cecaf608af61b0e1f6e612aede77 -> content 8c1275b150c0 (version v1)

live dev space: {'space_id': 'bbe0cecaf608af61b0e1f6e612aede77', 'title': 'FreshCart Sales & Stock Assistant [DEV]', 'parent_path': '/Workspace/Shared/genie/dev', 'etag': '1'}
dev reads: ['freshcart_dev.semantic.inventory_metrics', 'freshcart_dev.semantic.sales_metrics']


,record_id,kind,content_hash,version,actor,note
0,20261001T090700000000Z_deployed_8c1275b150c0,deployed,8c1275b150c0,v1,dana,


**What if a table were missing in the target catalog?** The deploy stops before touching the space. Here we pretend
the QA data pipeline hasn't created `dim_product` yet:

In [11]:
broken_qa = SimulatedWorkspace(LAB / "tmp_missing", qa.catalog, missing_objects={"freshcart_qa.gold.dim_product"})
try:
    ops.deploy(broken_qa, qa, S.load(SPACE_FILE), HistoryStore(broken_qa, qa.history_root), config=cfg)
except ops.DeployError as e:
    print("deploy refused:", e)

deploy refused: qa: objects missing in catalog freshcart_qa: table freshcart_qa.gold.dim_product. Deploy the data layer to this environment first.


### 6.4 · Dana improves the space in the dev UI, and every change gets a version
Developers keep working the way they like: in the Genie UI of the dev space. Here Dana adds a suggested question and
sharpens an instruction (`ui_edit` simulates clicking in the UI). The hourly **snapshot** job notices the change and
stores a new version in dev's history, with a readable change list.

In [12]:
DEV_ID = r["space_id"]

def add_question_and_clarify(space):
    space["config"]["sample_questions"].append({"id": "a" * 32, "question": ["Which stores opened in FY2025?"]})
    ti = space["instructions"]["text_instructions"][0]
    ti["content"] = [line.replace("Round percentages to one decimal place in the summary text.",
                                  "Round percentages to one decimal place and money to whole dollars in the summary text.") for line in ti["content"]]

dws.ui_edit(DEV_ID, add_question_and_clarify)
snap_good = ops.snapshot(dws, dev, dh, actor="snapshot-job")
print(snap_good["note"])
print("\n".join(snap_good["changes"]))

2 change(s) since 20261001T090700000000Z_deployed_8c1275b150c0
+ added sample question: Which stores opened in FY2025?
~ changed text instruction: About FreshCart and this agent FreshCart is a grocery retail...
           "- Say when a result covers comparable stores only.\n",
      -    "- Round percentages to one decimal place in the summary text.\n",
      +    "- Round percentages to one decimal place and money to whole dollars in the summary text.\n",
           "\n",


### 6.5 · A second edit, with a mistake nobody notices
Later, Dana edits the trusted example *"What were net sales and margin by region last week?"* and accidentally
changes `MEASURE(net_sales)` to `MEASURE(gross_sales)`. The SQL is perfectly valid, so nothing complains. The
snapshot job records it as another version:

In [13]:
def the_mistake(space):
    for e in space["instructions"]["example_question_sqls"]:
        if "net sales and margin by region" in S.sql_text(e["question"]):
            e["sql"] = [line.replace("MEASURE(net_sales)", "MEASURE(gross_sales)") for line in e["sql"]]

dws.ui_edit(DEV_ID, the_mistake)
snap_bad = ops.snapshot(dws, dev, dh, actor="snapshot-job")
print("\n".join(snap_bad["changes"]))
print("\nnothing changed since then ->", ops.snapshot(dws, dev, dh))
history("dev")

~ changed example SQL: What were net sales and margin by region last week?
           "SELECT region,\n",
      -    "       MEASURE(net_sales)        AS net_sales,\n",
      +    "       MEASURE(gross_sales)        AS net_sales,\n",
           "       MEASURE(gross_margin)     AS gross_margin,\n",

nothing changed since then -> None


,record_id,kind,content_hash,version,actor,note
0,20261001T090700000000Z_deployed_8c1275b150c0,deployed,8c1275b150c0,v1,dana,
1,20261001T091400000000Z_snapshot_46141340478f,snapshot,46141340478f,,snapshot-job,2 change(s) since 20261001T090700000000Z_deployed_8c1275b150c0
2,20261001T092100000000Z_snapshot_9b74ce4e32e5,snapshot,9b74ce4e32e5,,snapshot-job,1 change(s) since 20261001T091400000000Z_snapshot_46141340478f


Every state the dev space has been in is now a numbered version in its history: the first deploy, the good edit, the
mistake.

### 6.6 · The changes go to git through a pull request
The **dev-sync** job (or Dana, by hand) runs `pull`: export the dev space, neutralise the catalog, write the file in
canonical form. The pull request shows exactly what changed, in plain words and as a line diff:

In [14]:
git("checkout", "-q", "-b", "genie/dev-sync")
changes = ops.pull(dws, dev, SPACE_FILE)
print("what `pull` reports:"); print("\n".join(changes))
print("\nwhat the reviewer sees in the pull request (git diff):")
print(git("diff", "--stat", quiet=True))
print("\n".join(l for l in git("diff", "-U0", quiet=True).splitlines() if l[:1] in "+-" and not l.startswith(("+++", "---"))))

what `pull` reports:
+ added sample question: Which stores opened in FY2025?
~ changed text instruction: About FreshCart and this agent FreshCart is a grocery retail...
           "- Say when a result covers comparable stores only.\n",
      -    "- Round percentages to one decimal place in the summary text.\n",
      +    "- Round percentages to one decimal place and money to whole dollars in the summary text.\n",
           "\n",
~ changed example SQL: What were net sales and margin by region last week?
           "SELECT region,\n",
      -    "       MEASURE(net_sales)        AS net_sales,\n",
      +    "       MEASURE(gross_sales)        AS net_sales,\n",
           "       MEASURE(gross_margin)     AS gross_margin,\n",

what the reviewer sees in the pull request (git diff):
genie/space/freshcart_assistant.geniespace.json | 10 ++++++++--
 1 file changed, 8 insertions(+), 2 deletions(-)
+      {
+        "id": "aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa",
+        "question": [
+          "

`genie-ci` runs on the pull request. **Static validation and the SQL check both pass**: the SQL is valid, and it
returns numbers. This is exactly why a SQL check alone is not enough, and why there is a benchmark gate in QA.

In [15]:
pr_space = S.load(SPACE_FILE)
print("validation errors:", S.validate(pr_space, forbidden_catalogs=cfg.catalogs)[0] or "none")
print("SQL failures:", ops.check_sql_locally(pr_space) or "none")
git("commit", "-q", "-am", "Genie: sync changes made in the dev space")
git("checkout", "-q", "main"); git("merge", "-q", "--no-ff", "genie/dev-sync", "-m", "Merge pull request #12 from genie/dev-sync")
git("log", "--oneline", "--graph")

validation errors: none


SQL failures: none


*   66e6fed Merge pull request #12 from genie/dev-sync
|\  
| * 942d0c8 Genie: sync changes made in the dev space
|/  
* a5666fd Genie space v1: bootstrap from the reviewed design


### 6.7 · The release pipeline: QA catches the mistake
Merging to main starts **genie-release**. This function does exactly what the workflow's jobs do (Part 7), so we can
watch each stage: build one artifact → deploy to QA → benchmark gate → smoke test → approval → prod → smoke → tag.

In [16]:
def release(approve=True):
    sha = git("rev-parse", "--short", "HEAD", quiet=True)
    version = f"{clock().strftime('%Y%m%d.%H%M')}-{sha}"
    artifact = S.load(SPACE_FILE)                                      # build: the one artifact for every environment
    print(f"BUILD   version {version}, content {S.content_hash(artifact)}")
    errs = S.validate(artifact, forbidden_catalogs=cfg.catalogs)[0] + ops.check_sql_locally(artifact)
    if errs:
        print("BUILD   failed:", errs); return None
    print("QA      deploy");   ops.deploy(qws, qa, artifact, qh, config=cfg, version=version, git_sha=sha, actor="genie-release")
    print("QA      gate");     res = ops.gate(qws, qa, qh)
    for row in res["results"]:
        if row["assessment"] == "BAD":
            print(f"          BAD: {row['question']}\n               {row['reason']}")
    if not res["passed"]:
        print("STOP    the qa gate failed: prod is not touched"); return None
    print("QA      smoke");    ops.smoke(qws, qa)
    if not approve:
        print("WAIT    for approval of the prod deployment"); return version
    print("PROD    approved by Priya (required reviewer)")
    print("PROD    deploy");   ops.deploy(pws, prod, artifact, ph, config=cfg, version=version, git_sha=sha, actor="genie-release")
    print("PROD    smoke");    ok = ops.smoke(pws, prod)["passed"]
    if not ok:
        print("PROD    smoke failed: automatic rollback"); ops.rollback(pws, prod, ph, to="previous", config=cfg); return None
    git("tag", f"genie-prod-{version}"); print(f"TAG     genie-prod-{version}")
    return version

release()

BUILD   version 20261001.0928-66e6fed, content 9b74ce4e32e5


QA      deploy
  created qa space 82467724ca1fc8d3b20278168b0c8732 -> content 9b74ce4e32e5 (version 20261001.0928-66e6fed)
QA      gate


  benchmarks on qa: 3/4 graded correct, 1 bad, 13 need review -> FAIL: 1 benchmark(s) answered incorrectly (allowed 0); accuracy 75% is below 95%
          BAD: What were net sales and margin by region last week?
               result differs: expected [('Northeast', 5579.91, 2003.78, 0.3591061504576239), ('Canada', 2706.5, 1020.47, 0.3770441529650841)] got [('Northeast', 5673.2300000000005, 2003.78, 0.3591061504576239), ('Canada', 2758.45, 1020.47, 0.3770441529650841)]
STOP    the qa gate failed: prod is not touched


The gate compared Genie's answer to the trusted example question with the benchmark's correct SQL: the numbers
differ (gross vs net sales), so the answer is `BAD`, the gate fails, and **prod is never touched**. The other
benchmarks are `NEEDS_REVIEW` in the simulator; on Databricks, Genie grades them too.

### 6.8 · Undo in dev: go back to the version before the mistake
Dana opens dev's history, finds the snapshot from just before the mistake, and rolls dev back to it. The good edit
from 6.4 is kept; only the mistake disappears.

In [17]:
print("rolling dev back to", snap_good["record_id"])
ops.rollback(dws, dev, dh, to=snap_good["record_id"], config=cfg, actor="dana")
_, dev_now = ops.live_neutral(dws, dev)
print("dev is now identical to the good snapshot:", S.content_hash(dev_now) == snap_good["content_hash"])
history("dev")

rolling dev back to 20261001T091400000000Z_snapshot_46141340478f
  rolling dev back to 20261001T091400000000Z_snapshot_46141340478f (content 46141340478f)
  ! dev: overwriting 3 change(s) made outside the pipeline (kept in backup 20261001T094900000000Z_backup_9b74ce4e32e5)
  updated dev space bbe0cecaf608af61b0e1f6e612aede77 -> content 46141340478f
dev is now identical to the good snapshot: True


,record_id,kind,content_hash,version,actor,note
0,20261001T090700000000Z_deployed_8c1275b150c0,deployed,8c1275b150c0,v1,dana,
1,20261001T091400000000Z_snapshot_46141340478f,snapshot,46141340478f,,snapshot-job,2 change(s) since 20261001T090700000000Z_deployed_8c1275b150c0
2,20261001T092100000000Z_snapshot_9b74ce4e32e5,snapshot,9b74ce4e32e5,,snapshot-job,1 change(s) since 20261001T091400000000Z_snapshot_46141340478f
3,20261001T094900000000Z_backup_9b74ce4e32e5,backup,9b74ce4e32e5,,dana,live state before deploying 46141340478f
4,20261001T095600000000Z_rollback_46141340478f,rollback,46141340478f,,dana,rollback to 20261001T091400000000Z_snapshot_46141340478f


Note the `backup` record: before the rollback overwrote dev, its state (with the mistake) was saved. Even an undo can
be undone. (Dev is not protected, so the rollback is allowed to overwrite the UI edits; it reports how many.) Now the fix goes through the same path: pull → pull request → merge → release.

In [18]:
git("checkout", "-q", "-b", "genie/fix-net-sales")
print("\n".join(ops.pull(dws, dev, SPACE_FILE)))
git("commit", "-q", "-am", "Genie: restore net sales in the region example")
git("checkout", "-q", "main"); git("merge", "-q", "--no-ff", "genie/fix-net-sales", "-m", "Merge pull request #13 from genie/fix-net-sales")
first_release = release()

~ changed example SQL: What were net sales and margin by region last week?
           "SELECT region,\n",
      -    "       MEASURE(gross_sales)        AS net_sales,\n",
      +    "       MEASURE(net_sales)        AS net_sales,\n",
           "       MEASURE(gross_margin)     AS gross_margin,\n",


BUILD   version 20261001.1003-41a0d98, content 46141340478f


QA      deploy
  updated qa space 82467724ca1fc8d3b20278168b0c8732 -> content 46141340478f (version 20261001.1003-41a0d98)
QA      gate


  benchmarks on qa: 4/4 graded correct, 0 bad, 13 need review -> PASS
QA      smoke
  smoke test on qa: 'What were net sales and margin by region last week?' -> COMPLETED with SQL -> PASS
PROD    approved by Priya (required reviewer)
PROD    deploy
  created prod space c9c85e37939b44d259742956a97cd14c -> content 46141340478f (version 20261001.1003-41a0d98)
PROD    smoke
  smoke test on prod: 'What were net sales and margin by region last week?' -> COMPLETED with SQL -> PASS
TAG     genie-prod-20261001.1003-41a0d98


This time QA passes, a reviewer approves, the **same artifact** is deployed to prod, the smoke test passes, and the
release is tagged in git.

In [19]:
git("tag", "-l", "genie-prod-*")
history("prod")

genie-prod-20261001.1003-41a0d98


,record_id,kind,content_hash,version,actor,note
0,20261001T103100000000Z_deployed_46141340478f,deployed,46141340478f,20261001.1003-41a0d98,genie-release,


### 6.9 · Someone edits prod directly in the UI: drift
A well-meaning analyst adds an instruction straight into the prod space. Prod is **protected**, so the next deploy
won't silently wipe that edit out. `plan` shows it, and `deploy` refuses, without writing anything, until someone
decides.

In [20]:
PROD_ID = ops.find_space(pws, prod)["space_id"]
pws.ui_edit(PROD_ID, lambda s: add_line(s["instructions"]["text_instructions"][0]["content"], "Always mention the weather."))

p = ops.plan(pws, prod, S.load(SPACE_FILE), ph)
print("plan:", p["action"]); print("drift (edited outside the pipeline):"); print("\n".join(p["drift"]))
try:
    ops.deploy(pws, prod, S.load(SPACE_FILE), ph, config=cfg, version="redeploy")
except ops.DriftError as e:
    print("\ndeploy refused:", e)

plan: update
drift (edited outside the pipeline):
~ changed text instruction: About FreshCart and this agent FreshCart is a grocery retail...
           "- What were net sales and margin by region last week? Last week always means the last completed fiscal week, Sunday to Saturday.\n",
      -    "- How are we trading year to date versus last year? Total business comparison. For comparable-store growth use fn_like_for_like_sales."
      +    "- How are we trading year to date versus last year? Total business comparison. For comparable-store growth use fn_like_for_like_sales.\n",
      +    "Always mention the weather."
         ],

deploy refused: prod: the live space was edited outside the pipeline since the last deployment (1 change(s)). Pull it into git, or redeploy with --allow-drift to overwrite.


Two ways out, both safe:

| Option | When | How |
|---|---|---|
| **Adopt** the edit | it's a good change | `pull --env prod` into a branch → pull request → normal release |
| **Overwrite** it | it shouldn't be there | run the release with *allow drift* (workflow input `allow_drift`), the edit is kept in a `backup` record |

Here the edit shouldn't be there, so we overwrite it. The backup keeps it:

In [21]:
ops.deploy(pws, prod, S.load(SPACE_FILE), ph, config=cfg, version=first_release, actor="priya", allow_drift=True)
backup = ph.latest("prod", ("backup",))
print("the analyst's edit is preserved in", backup["record_id"], ":", any("weather" in l for l in backup["space"]["instructions"]["text_instructions"][0]["content"]))

  ! prod: overwriting 1 change(s) made outside the pipeline (kept in backup 20261001T103800000000Z_backup_0a9944933f8d)
  updated prod space c9c85e37939b44d259742956a97cd14c -> content 46141340478f (version 20261001.1003-41a0d98)
the analyst's edit is preserved in 20261001T103800000000Z_backup_0a9944933f8d : True


### 6.10 · An incident: a release that the benchmarks couldn't catch
Next sprint, a change to the instructions ("answer in one sentence") passes every gate: the SQL answers are still
right. But on Monday, business users complain that answers lack the context they need. Restoring the previous
release is one command (in GitHub: the **genie-rollback** workflow, `environment = prod`, `to = previous`).

In [22]:
git("checkout", "-q", "-b", "genie/terse")
dws.ui_edit(DEV_ID, lambda s: add_line(s["instructions"]["text_instructions"][0]["content"], "Answer in one sentence, no explanations."))
ops.pull(dws, dev, SPACE_FILE)
git("commit", "-q", "-am", "Genie: shorter answers"); git("checkout", "-q", "main")
git("merge", "-q", "--no-ff", "genie/terse", "-m", "Merge pull request #14 from genie/terse")
second_release = release()

BUILD   version 20261001.1052-93e48ea, content 54f5e5b49886


QA      deploy
  updated qa space 82467724ca1fc8d3b20278168b0c8732 -> content 54f5e5b49886 (version 20261001.1052-93e48ea)
QA      gate


  benchmarks on qa: 4/4 graded correct, 0 bad, 13 need review -> PASS
QA      smoke
  smoke test on qa: 'What were net sales and margin by region last week?' -> COMPLETED with SQL -> PASS
PROD    approved by Priya (required reviewer)
PROD    deploy
  updated prod space c9c85e37939b44d259742956a97cd14c -> content 54f5e5b49886 (version 20261001.1052-93e48ea)
PROD    smoke
  smoke test on prod: 'What were net sales and margin by region last week?' -> COMPLETED with SQL -> PASS
TAG     genie-prod-20261001.1052-93e48ea


In [23]:
print("Monday 08:55: complaints. Roll back prod to the previous release.\n")
ops.rollback(pws, prod, ph, to="previous", config=cfg, actor="priya")
_, prod_now = ops.live_neutral(pws, prod)
first = next(r for r in ph.records("prod", ("deployed",)) if r["version"] == first_release)
print("\nprod content is back to the first release:", S.content_hash(prod_now) == first["content_hash"])
print("same space id (users keep their conversations):", ops.find_space(pws, prod)["space_id"] == PROD_ID)
smoke = ops.smoke(pws, prod)

Monday 08:55: complaints. Roll back prod to the previous release.

  rolling prod back to 20261001T104500000000Z_deployed_46141340478f (content 46141340478f)
  updated prod space c9c85e37939b44d259742956a97cd14c -> content 46141340478f (version 20261001.1003-41a0d98)

prod content is back to the first release: True
same space id (users keep their conversations): True


  smoke test on prod: 'What were net sales and margin by region last week?' -> COMPLETED with SQL -> PASS


### 6.11 · Roll back to any git version
Every prod release is tagged in git, so you can also return to any tagged release, or any commit, by its git name:

In [24]:
tag = git("tag", "-l", "genie-prod-*", quiet=True).splitlines()[0]
print("deploying", tag, "to qa")
rec = ops.rollback(qws, qa, qh, to=f"git:{tag}", config=cfg, actor="priya")
print("qa now runs", rec["version"], "content", rec["content_hash"])

deploying genie-prod-20261001.1003-41a0d98 to qa
  rolling qa back to git:genie-prod-20261001.1003-41a0d98 (content 46141340478f)
  updated qa space 82467724ca1fc8d3b20278168b0c8732 -> content 46141340478f (version genie-prod-20261001.1003-41a0d98)
qa now runs genie-prod-20261001.1003-41a0d98 content 46141340478f


### 6.12 · The audit trail
Every environment's history answers *who changed what, when, and from which commit*. Each record also contains the
full space, so any of them can be redeployed.

In [25]:
display(history("prod"))
display(history("qa"))

,record_id,kind,content_hash,version,actor,note
0,20261001T103100000000Z_deployed_46141340478f,deployed,46141340478f,20261001.1003-41a0d98,genie-release,
1,20261001T103800000000Z_backup_0a9944933f8d,backup,0a9944933f8d,,priya,live state before deploying 20261001.1003-41a0d98
2,20261001T104500000000Z_deployed_46141340478f,deployed,46141340478f,20261001.1003-41a0d98,priya,
3,20261001T112000000000Z_backup_46141340478f,backup,46141340478f,,genie-release,live state before deploying 20261001.1052-93e48ea
4,20261001T112700000000Z_deployed_54f5e5b49886,deployed,54f5e5b49886,20261001.1052-93e48ea,genie-release,
5,20261001T113400000000Z_backup_54f5e5b49886,backup,54f5e5b49886,,priya,live state before deploying 20261001.1003-41a0d98
6,20261001T114100000000Z_rollback_46141340478f,rollback,46141340478f,20261001.1003-41a0d98,priya,rollback to 20261001T104500000000Z_deployed_46141340478f


,record_id,kind,content_hash,version,actor,note
0,20261001T093500000000Z_deployed_9b74ce4e32e5,deployed,9b74ce4e32e5,20261001.0928-66e6fed,genie-release,
1,20261001T094200000000Z_gate_9b74ce4e32e5,gate,9b74ce4e32e5,20261001.0928-66e6fed,,"accuracy 75%, bad 1"
2,20261001T101000000000Z_backup_9b74ce4e32e5,backup,9b74ce4e32e5,,genie-release,live state before deploying 20261001.1003-41a0d98
3,20261001T101700000000Z_deployed_46141340478f,deployed,46141340478f,20261001.1003-41a0d98,genie-release,
4,20261001T102400000000Z_gate_46141340478f,gate,46141340478f,20261001.1003-41a0d98,,"accuracy 100%, bad 0"
5,20261001T105900000000Z_backup_46141340478f,backup,46141340478f,,genie-release,live state before deploying 20261001.1052-93e48ea
6,20261001T110600000000Z_deployed_54f5e5b49886,deployed,54f5e5b49886,20261001.1052-93e48ea,genie-release,
7,20261001T111300000000Z_gate_54f5e5b49886,gate,54f5e5b49886,20261001.1052-93e48ea,,"accuracy 100%, bad 0"
8,20261001T114800000000Z_backup_54f5e5b49886,backup,54f5e5b49886,,priya,live state before deploying genie-prod-20261001.1003-41a0d98
9,20261001T115500000000Z_rollback_46141340478f,rollback,46141340478f,genie-prod-20261001.1003-41a0d98,priya,rollback to git:genie-prod-20261001.1003-41a0d98


In [26]:
print("What changed between the two prod releases:")
releases = ph.records("prod", ("deployed",))
rel = {r["version"]: r for r in releases}
print("\n".join(S.diff(rel[first_release]["space"], rel[second_release]["space"])))
print("\nGit history of the space file:")
git("log", "--oneline", "--decorate", "--", "genie/space/freshcart_assistant.geniespace.json")

What changed between the two prod releases:
~ changed text instruction: About FreshCart and this agent FreshCart is a grocery retail...
           "- What were net sales and margin by region last week? Last week always means the last completed fiscal week, Sunday to Saturday.\n",
      -    "- How are we trading year to date versus last year? Total business comparison. For comparable-store growth use fn_like_for_like_sales."
      +    "- How are we trading year to date versus last year? Total business comparison. For comparable-store growth use fn_like_for_like_sales.\n",
      +    "Answer in one sentence, no explanations."
         ],

Git history of the space file:
bb4ffcc (genie/terse) Genie: shorter answers
84cb491 (genie/fix-net-sales) Genie: restore net sales in the region example
942d0c8 (genie/dev-sync) Genie: sync changes made in the dev space
a5666fd Genie space v1: bootstrap from the reviewed design


### 6.13 · Etags: two people saving at the same time
If the space changes between the moment the pipeline reads it and the moment it writes, the stale etag makes the
update fail instead of overwriting the other person's edit:

In [27]:
from genie_cicd.workspace import ConflictError
read = dws.get_space(DEV_ID)                                       # the pipeline reads (etag e1)
dws.ui_edit(DEV_ID, lambda s: None)                                # meanwhile someone saves in the UI (etag e2)
try:
    dws.update_space(DEV_ID, title=dev.title, description=dev.description, warehouse_id=dev.warehouse_id,
                     serialized_space=read["serialized_space"], etag=read["etag"])
except ConflictError as e:
    print("update rejected:", e)

update rejected: etag 5 is stale; the space is now at 6


### 6.14 · The same thing from the command line
The workflows call the CLI. Here are the commands, run against a fresh simulated setup (`--target sim`). Against
Databricks, drop `--target sim`.

In [28]:
SIM = LAB / "cli_sim"
def cli(*args):
    cmd = [sys.executable, "-m", "genie_cicd", "--target", "sim", "--sim-dir", str(SIM), *args]
    print("$ python -m genie_cicd", " ".join(args))
    p = subprocess.run(cmd, capture_output=True, text=True)
    print((p.stdout + p.stderr).rstrip()); print(f"(exit code {p.returncode})\n")

cli("validate")
cli("deploy", "--env", "dev", "--version", "demo-1")
cli("promote", "--from", "dev", "--to", "qa")
cli("gate", "--env", "qa")
cli("promote", "--from", "qa", "--to", "prod")
cli("smoke", "--env", "prod")
cli("plan", "--env", "prod")
cli("history", "--env", "prod")

$ python -m genie_cicd validate


space content 8c1275b150c0: {'table': 2, 'metric view': 2, 'sample question': 4, 'text instruction': 1, 'example SQL': 6, 'trusted function': 1, 'SQL filter': 2, 'SQL expression': 1, 'benchmark': 17}
errors:
  (none)
warnings:
  (none)
(exit code 0)

$ python -m genie_cicd deploy --env dev --version demo-1


  created dev space bbe0cecaf608af61b0e1f6e612aede77 -> content 8c1275b150c0 (version demo-1)
(exit code 0)

$ python -m genie_cicd promote --from dev --to qa


  created qa space 82467724ca1fc8d3b20278168b0c8732 -> content 8c1275b150c0 (version demo-1)
(exit code 0)

$ python -m genie_cicd gate --env qa


  benchmarks on qa: 4/4 graded correct, 0 bad, 13 need review -> PASS
    (13 benchmark(s) need human review; --verbose lists them)
(exit code 0)

$ python -m genie_cicd promote --from qa --to prod
  created prod space c9c85e37939b44d259742956a97cd14c -> content 8c1275b150c0 (version demo-1)
(exit code 0)

$ python -m genie_cicd smoke --env prod


  smoke test on prod: 'What were net sales and margin by region last week?' -> COMPLETED with SQL -> PASS
(exit code 0)

$ python -m genie_cicd plan --env prod
prod: noop (live 8c1275b150c0 -> new 8c1275b150c0)
changes:
  (none)
drift since the last release (edits made outside the pipeline):
  (none)
(exit code 0)

$ python -m genie_cicd history --env prod


record_id                                            kind      version                    actor        note
20261001T200129694610Z_deployed_8c1275b150c0         deployed  demo-1                     unknown      promoted from qa record 20261001T200128683839Z_deployed_8c1275b150c0
(exit code 0)



---
# 7 · The CI/CD pipeline on GitHub Actions

Four workflows, all in `.github/workflows/`, all calling `python -m genie_cicd`.

| Workflow | Trigger | What it does | Needs Databricks? |
|---|---|---|---|
| **genie-ci** | every pull request touching the space or the tooling | validate, run every SQL, run the tests, post the change list | no |
| **genie-release** | merge to `main` that changes the space file; or manual | build one artifact → **qa**: plan, deploy, benchmark gate, smoke → **prod** (after approval): plan, deploy, smoke (auto-rollback on failure), git tag | yes |
| **genie-rollback** | manual (Actions tab) | roll an environment back to `previous`, a history record, or `git:<ref>`; smoke test | yes |
| **genie-dev-sync** | hourly; or manual | snapshot dev into its history, export it, open/update a sync pull request | yes |

Until the GitHub environments have their Databricks variables, the Databricks steps are skipped with a notice, so the
workflows are safe in a repository that isn't connected yet.

### The rules that make it safe

| Rule | Where it is enforced |
|---|---|
| Nothing reaches `main` without review and green checks | branch protection on `main`: require a pull request, 1+ approval, status check **genie-ci / checks** |
| Only reviewed content is released | genie-release runs only from `main` |
| What was tested is what ships | one artifact built in `build`, downloaded by `qa` and `prod` |
| Prod only after QA passed | `prod` job: `needs: qa` and `if: needs.qa.outputs.gated == 'true'` |
| A human approves prod | GitHub environment `prod` with **required reviewers** (and "deployment branches: main") |
| One deployment at a time | `concurrency: genie-release` shared by release and rollback |
| No secrets in GitHub | GitHub OIDC → Databricks workload identity federation (`DATABRICKS_AUTH_TYPE: github-oidc`) |
| Prod failures are reverted fast | smoke test failure triggers `rollback --to previous` in the same job |
| Every release is traceable | git tag `genie-prod-<version>` + `deployed` record with version, commit and actor |

<details>
<summary><b>📄 genie-ci: <code>.github/workflows/genie-ci.yml</code></b> (click to open)</summary>

```yaml
# Genie space: checks on every pull request (no Databricks access needed)
#
#  1. the space file is valid, canonical and environment-neutral, and every SQL in it runs
#  2. the version-control / promotion / rollback logic passes its tests (simulated dev, qa, prod)
#  3. reviewers get a readable list of what changed in the space (job summary)
#
# Make the "genie-ci / checks" job a required status check on main (branch protection), so no
# change to the space can merge without passing it and without a reviewer's approval.
name: genie-ci

on:
  pull_request:
    paths:
      - "genie/**"
      - "genie_cicd/**"
      - "deploy/genie/**"
      - "semantic/**"
      - "tests/test_genie_cicd.py"
      - ".github/workflows/genie-*.yml"
  push:
    branches: [main]
    paths:
      - "genie/**"
      - "genie_cicd/**"
      - "deploy/genie/**"

permissions:
  contents: read

jobs:
  checks:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
        with:
          fetch-depth: 0                       # the base branch is needed for the change list

      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"

      - run: pip install -r deploy/genie/requirements.txt

      - name: Validate the space file (static checks + every SQL on the local warehouse)
        run: python -m genie_cicd validate --sql

      - name: Version-control, promotion and rollback tests
        run: python tests/run_tests.py

      - name: What changed in the Genie space
        if: github.event_name == 'pull_request'
        run: |
          git show "origin/${{ github.base_ref }}:genie/space/freshcart_assistant.geniespace.json" > /tmp/base.json 2>/dev/null || echo '{}' > /tmp/base.json
          python -m genie_cicd diff --base /tmp/base.json
```
</details>

<details>
<summary><b>📄 genie-release: <code>.github/workflows/genie-release.yml</code></b> (click to open)</summary>

```yaml
# Genie space: release pipeline  main -> qa (benchmark gate) -> prod (manual approval)
#
# Build once, promote the same artifact: the exact file that passed QA is the file that reaches prod.
#
# One-time setup (see notebooks/Genie_Version_Control_and_CICD.ipynb, part 7):
#   GitHub environments `qa` and `prod`; on `prod` add required reviewers.
#   Variables in each environment:
#     DATABRICKS_HOST          workspace URL of that environment
#     DATABRICKS_CLIENT_ID     application id of the environment's deployment service principal
#     GENIE_WAREHOUSE_ID       SQL warehouse the space uses
#     GENIE_SPACE_ID           (optional) id of the existing space; found by title when empty
#   Authentication is GitHub OIDC -> Databricks workload identity federation: no secrets are stored.
#   (Alternative: OAuth M2M with a DATABRICKS_CLIENT_SECRET secret and DATABRICKS_AUTH_TYPE=oauth-m2m.)
#
# While the variables are not set, the deploy steps are skipped with a notice, so the workflow is
# safe to have in a repository that is not connected to Databricks yet.
name: genie-release

on:
  push:
    branches: [main]
    paths:
      - "genie/space/**"
      - "deploy/genie/environments.yml"
  workflow_dispatch:
    inputs:
      allow_drift:
        description: Overwrite edits made in the qa/prod UI since the last release (a backup is kept)
        type: boolean
        default: false

concurrency:
  group: genie-release                         # one release at a time; never cancel a deploy halfway
  cancel-in-progress: false

permissions:
  contents: write                              # push the release tag
  id-token: write                              # GitHub OIDC token for Databricks

jobs:
  build:
    runs-on: ubuntu-latest
    outputs:
      version: ${{ steps.version.outputs.version }}
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
      - run: pip install -r deploy/genie/requirements.txt
      - name: Validate
        run: python -m genie_cicd validate --sql
      - id: version
        name: Version = UTC date + commit
        run: echo "version=$(date -u +%Y%m%d.%H%M)-${GITHUB_SHA::7}" >> "$GITHUB_OUTPUT"
      - name: Package the release artifact
        run: |
          mkdir -p dist
          cp genie/space/freshcart_assistant.geniespace.json dist/
          python - <<'EOF'
          import json
          from genie_cicd import spec as S
          s = S.load("dist/freshcart_assistant.geniespace.json")
          json.dump({"version": "${{ steps.version.outputs.version }}", "git_sha": "${{ github.sha }}",
                     "content_hash": S.content_hash(s), "summary": S.summary(s)}, open("dist/manifest.json", "w"), indent=1)
          EOF
          cat dist/manifest.json
      - uses: actions/upload-artifact@v4
        with:
          name: genie-space
          path: dist/

  qa:
    needs: build
    runs-on: ubuntu-latest
    environment: qa
    outputs:
      gated: ${{ steps.gate.outcome == 'success' }}
    env:
      DATABRICKS_HOST: ${{ vars.DATABRICKS_HOST }}
      DATABRICKS_CLIENT_ID: ${{ vars.DATABRICKS_CLIENT_ID }}
      DATABRICKS_AUTH_TYPE: github-oidc
      GENIE_QA_WAREHOUSE_ID: ${{ vars.GENIE_WAREHOUSE_ID }}
      GENIE_QA_SPACE_ID: ${{ vars.GENIE_SPACE_ID }}
      VERSION: ${{ needs.build.outputs.version }}
      DRIFT_FLAG: ${{ inputs.allow_drift && '--allow-drift' || '' }}
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
      - run: pip install -r deploy/genie/requirements.txt
      - uses: actions/download-artifact@v4
        with:
          name: genie-space
          path: dist/
      - id: cfg
        name: Is the qa environment connected to Databricks?
        run: |
          if [ -z "$DATABRICKS_HOST" ]; then
            echo "::notice::qa is not configured (no DATABRICKS_HOST variable); skipping the deployment"
            echo "ready=false" >> "$GITHUB_OUTPUT"
          else
            echo "ready=true" >> "$GITHUB_OUTPUT"
          fi
      - name: Plan
        if: steps.cfg.outputs.ready == 'true'
        run: python -m genie_cicd plan --env qa --file dist/freshcart_assistant.geniespace.json
      - name: Deploy to qa
        if: steps.cfg.outputs.ready == 'true'
        run: python -m genie_cicd deploy --env qa --file dist/freshcart_assistant.geniespace.json --version "$VERSION" --git-sha "$GITHUB_SHA" $DRIFT_FLAG
      - id: gate
        name: Benchmark gate
        if: steps.cfg.outputs.ready == 'true'
        run: python -m genie_cicd gate --env qa
      - name: Smoke test
        if: steps.cfg.outputs.ready == 'true'
        run: python -m genie_cicd smoke --env qa

  prod:
    needs: [build, qa]
    if: needs.qa.outputs.gated == 'true'      # never reaches prod without a passed qa gate
    runs-on: ubuntu-latest
    environment: prod                          # required reviewers approve here
    env:
      DATABRICKS_HOST: ${{ vars.DATABRICKS_HOST }}
      DATABRICKS_CLIENT_ID: ${{ vars.DATABRICKS_CLIENT_ID }}
      DATABRICKS_AUTH_TYPE: github-oidc
      GENIE_PROD_WAREHOUSE_ID: ${{ vars.GENIE_WAREHOUSE_ID }}
      GENIE_PROD_SPACE_ID: ${{ vars.GENIE_SPACE_ID }}
      VERSION: ${{ needs.build.outputs.version }}
      DRIFT_FLAG: ${{ inputs.allow_drift && '--allow-drift' || '' }}
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
      - run: pip install -r deploy/genie/requirements.txt
      - uses: actions/download-artifact@v4
        with:
          name: genie-space
          path: dist/
      - name: Plan (shows drift if someone edited prod in the UI)
        run: python -m genie_cicd plan --env prod --file dist/freshcart_assistant.geniespace.json
      - name: Deploy to prod
        run: python -m genie_cicd deploy --env prod --file dist/freshcart_assistant.geniespace.json --version "$VERSION" --git-sha "$GITHUB_SHA" $DRIFT_FLAG
      - name: Smoke test (rolls back automatically if it fails)
        run: |
          if ! python -m genie_cicd smoke --env prod; then
            echo "::error::smoke test failed on prod; rolling back to the previous release"
            python -m genie_cicd rollback --env prod --to previous
            exit 1
          fi
      - name: Tag the release
        run: |
          git tag "genie-prod-$VERSION" "$GITHUB_SHA"
          git push origin "genie-prod-$VERSION"
```
</details>

<details>
<summary><b>📄 genie-rollback: <code>.github/workflows/genie-rollback.yml</code></b> (click to open)</summary>

```yaml
# Genie space: rollback (run manually from the Actions tab)
#
#   to = previous                 the release before the one that is live now (run again to go further back)
#   to = <history record id>      any backup / snapshot / release in the environment's history
#   to = git:<tag or commit>      the space file as it was in git, e.g. git:genie-prod-20261001.0915-1a2b3c4
#
# The `prod` GitHub environment's required reviewers approve the rollback like any deploy.
# A backup of the live space is taken first, so a rollback can itself be rolled back.
name: genie-rollback

on:
  workflow_dispatch:
    inputs:
      environment:
        description: Environment to roll back
        type: choice
        options: [prod, qa, dev]
        required: true
      to:
        description: "previous | <history record id> | git:<tag or commit>"
        default: previous
        required: true
      reason:
        description: Why (kept in the run log)
        required: true

concurrency:
  group: genie-release                         # never overlaps with a release
  cancel-in-progress: false

permissions:
  contents: read
  id-token: write

jobs:
  rollback:
    runs-on: ubuntu-latest
    environment: ${{ inputs.environment }}
    env:
      DATABRICKS_HOST: ${{ vars.DATABRICKS_HOST }}
      DATABRICKS_CLIENT_ID: ${{ vars.DATABRICKS_CLIENT_ID }}
      DATABRICKS_AUTH_TYPE: github-oidc
      GENIE_DEV_WAREHOUSE_ID: ${{ vars.GENIE_WAREHOUSE_ID }}
      GENIE_QA_WAREHOUSE_ID: ${{ vars.GENIE_WAREHOUSE_ID }}
      GENIE_PROD_WAREHOUSE_ID: ${{ vars.GENIE_WAREHOUSE_ID }}
      GENIE_DEV_SPACE_ID: ${{ vars.GENIE_SPACE_ID }}
      GENIE_QA_SPACE_ID: ${{ vars.GENIE_SPACE_ID }}
      GENIE_PROD_SPACE_ID: ${{ vars.GENIE_SPACE_ID }}
      TARGET_ENV: ${{ inputs.environment }}      # inputs reach the shell as variables, never as code
      ROLLBACK_TO: ${{ inputs.to }}
      REASON: ${{ inputs.reason }}
    steps:
      - uses: actions/checkout@v4
        with:
          fetch-depth: 0                       # tags and history for git:<ref> targets
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
      - run: pip install -r deploy/genie/requirements.txt
      - name: Reason
        run: |
          echo "Rollback of $TARGET_ENV to '$ROLLBACK_TO' requested by $GITHUB_ACTOR"
          echo "Reason: $REASON"
      - name: History before
        run: python -m genie_cicd history --env "$TARGET_ENV"
      - name: Roll back
        run: python -m genie_cicd rollback --env "$TARGET_ENV" --to "$ROLLBACK_TO"
      - name: Smoke test
        if: inputs.environment != 'dev'
        run: python -m genie_cicd smoke --env "$TARGET_ENV"
```
</details>

<details>
<summary><b>📄 genie-dev-sync: <code>.github/workflows/genie-dev-sync.yml</code></b> (click to open)</summary>

```yaml
# Genie space: capture every change made in the dev UI
#
# Every hour (and on demand):
#   1. snapshot  - if the dev space changed, store a new version in dev's history on the UC volume
#                  (this is the undo history for dev: `rollback --env dev --to <record id>`)
#   2. pull      - export the dev space into the git file
#   3. if the file changed, open (or update) the pull request "Genie: sync changes made in dev",
#      which goes through review, genie-ci and then the release pipeline like any other change.
#
# Note: pull requests opened with the default GITHUB_TOKEN do not trigger other workflows. To have
# genie-ci run on this PR automatically, use a GitHub App token or a fine-grained PAT stored as the
# secret GENIE_SYNC_TOKEN (used below when present).
name: genie-dev-sync

on:
  schedule:
    - cron: "17 * * * *"
  workflow_dispatch:

concurrency:
  group: genie-dev-sync
  cancel-in-progress: false

permissions:
  contents: write
  pull-requests: write
  id-token: write

jobs:
  sync:
    runs-on: ubuntu-latest
    environment: dev
    env:
      DATABRICKS_HOST: ${{ vars.DATABRICKS_HOST }}
      DATABRICKS_CLIENT_ID: ${{ vars.DATABRICKS_CLIENT_ID }}
      DATABRICKS_AUTH_TYPE: github-oidc
      GENIE_DEV_WAREHOUSE_ID: ${{ vars.GENIE_WAREHOUSE_ID }}
      GENIE_DEV_SPACE_ID: ${{ vars.GENIE_SPACE_ID }}
      GH_TOKEN: ${{ secrets.GENIE_SYNC_TOKEN || github.token }}
    steps:
      - uses: actions/checkout@v4
        with:
          token: ${{ secrets.GENIE_SYNC_TOKEN || github.token }}
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
      - run: pip install -r deploy/genie/requirements.txt
      - id: cfg
        run: |
          if [ -z "$DATABRICKS_HOST" ]; then
            echo "::notice::dev is not configured (no DATABRICKS_HOST variable); nothing to sync"
            echo "ready=false" >> "$GITHUB_OUTPUT"
          else
            echo "ready=true" >> "$GITHUB_OUTPUT"
          fi
      - name: Snapshot dev into its version history
        if: steps.cfg.outputs.ready == 'true'
        run: python -m genie_cicd snapshot --env dev
      - name: Pull dev into git
        if: steps.cfg.outputs.ready == 'true'
        run: python -m genie_cicd pull --env dev
      - name: Open or update the sync pull request
        if: steps.cfg.outputs.ready == 'true'
        run: |
          FILE=genie/space/freshcart_assistant.geniespace.json
          if git diff --quiet -- "$FILE"; then echo "dev matches main"; exit 0; fi
          python -m genie_cicd validate || echo "::warning::the dev space has validation errors; see the PR checks"
          BRANCH=genie/dev-sync
          git config user.name "genie-dev-sync"
          git config user.email "genie-dev-sync@users.noreply.github.com"
          git checkout -B "$BRANCH"
          git add "$FILE"
          git commit -m "Genie: sync changes made in the dev space"
          git push --force origin "$BRANCH"
          BODY="Changes made in the dev Genie space UI, exported by genie-dev-sync. Review the diff, then merge to release to qa and prod."
          gh pr view "$BRANCH" >/dev/null 2>&1 || gh pr create --base main --head "$BRANCH" --title "Genie: sync changes made in dev" --body "$BODY"
```
</details>

### Branching and release flow

```
 feature branch / genie/dev-sync ──PR──► main ──(genie-release)──► qa ──gate──► [approve] ──► prod ──► tag genie-prod-<version>
        ▲                                   │
        └── fix forward with a new PR ◄─────┘  or  genie-rollback (prod, to=previous) for an immediate revert
```

* **Fix forward** (normal): make the correction in dev, pull, PR, release.
* **Roll back** (emergency): genie-rollback restores the last good release in about a minute; then fix forward.

---
# 8 · Going live on Databricks

A checklist for the one-time setup. Commands use the Databricks CLI and SQL; adapt names to your organisation.

### 8.1 · Decide how environments are separated
| Option | Recommended for |
|---|---|
| **Separate workspaces** for dev, qa and prod (each with its own catalog: `freshcart_dev`, `freshcart_qa`, `freshcart`) | production use: separate permissions, compute and blast radius |
| One workspace, **separate catalogs** per environment | smaller teams; still fully supported by `environments.yml` (same host, different catalogs) |

### 8.2 · Each environment needs its own copy of the data layer
The space reads `{{catalog}}.gold.*` and `{{catalog}}.semantic.*`, so every environment's catalog must contain the gold
tables, the two metric views and the like-for-like function (see `databricks/` and `docs/09-run-on-databricks.md`; run
them with the catalog name of the environment). `deploy` checks every object exists before touching the space, so a
missing table stops the release with a clear message instead of breaking the assistant.

### 8.3 · A volume for the version history, in each environment
```sql
CREATE SCHEMA IF NOT EXISTS freshcart_qa.ops COMMENT 'Operational data: Genie version history';
CREATE VOLUME IF NOT EXISTS freshcart_qa.ops.genie_history COMMENT 'Genie space snapshots, backups and releases';
-- same for freshcart_dev.ops and freshcart.ops
```

### 8.4 · One deployment service principal per environment, with least privilege
| The service principal needs | Why |
|---|---|
| `CAN_USE` on the environment's SQL warehouse | the space is attached to it; benchmarks run on it |
| `USE CATALOG`, `USE SCHEMA`, `SELECT` on the gold tables, metric views; `EXECUTE` on the function | dependency check and benchmark runs |
| `READ VOLUME`, `WRITE VOLUME` on `<catalog>.ops.genie_history` | the version history |
| `CAN_MANAGE` on the workspace folder `/Workspace/Shared/genie/<env>` | create and update the space there |

| People | dev | qa | prod |
|---|---|---|---|
| Genie developers | `CAN_EDIT` / `CAN_MANAGE` on the space | `CAN_RUN` (test) | `CAN_RUN` |
| Business users | none | testers: `CAN_RUN` | `CAN_RUN` |
| Only the service principal | | `CAN_MANAGE` | `CAN_MANAGE` |

Nobody but the pipeline can edit qa or prod, so drift should only come from admins in an emergency, and is then caught.
Remember Genie queries **as the end user**: business users need `SELECT` on the data (row filters still apply).
Space permissions are set once in the UI (Share) and are not part of the serialized space.

### 8.5 · Connect GitHub to Databricks without secrets (workload identity federation)
1. For each environment's service principal, create a **federation policy** trusting GitHub:
   issuer `https://token.actions.githubusercontent.com`, subject `repo:<org>/<repo>:environment:<env>`
   (only workflows running in that GitHub environment can use it). Use the audience given in the Databricks guide
   *"Enable workload identity federation for GitHub Actions"* (`databricks account service-principal-federation-policy create ...`).
2. In GitHub → Settings → Environments, create **dev**, **qa**, **prod** with these **variables**:
   `DATABRICKS_HOST`, `DATABRICKS_CLIENT_ID` (the service principal's application id), `GENIE_WAREHOUSE_ID`,
   optionally `GENIE_SPACE_ID`.
3. On **prod**: *Required reviewers* (e.g. the analytics lead) and *Deployment branches: main*. Optionally a wait timer.
4. Branch protection on `main`: require pull requests, 1+ approval, and the **genie-ci / checks** status check.
5. Workflows already request `permissions: id-token: write` and set `DATABRICKS_AUTH_TYPE: github-oidc`; the Databricks
   SDK does the token exchange. (No federation? Use OAuth M2M: a `DATABRICKS_CLIENT_SECRET` environment secret and
   `DATABRICKS_AUTH_TYPE: oauth-m2m`.)

### 8.6 · First release
```bash
# on a laptop, one profile per environment (names match environments.yml)
databricks auth login --host https://<dev-workspace>  --profile freshcart-dev
databricks auth login --host https://<qa-workspace>   --profile freshcart-qa
export GENIE_DEV_WAREHOUSE_ID=<id> GENIE_QA_WAREHOUSE_ID=<id> GENIE_PROD_WAREHOUSE_ID=<id>

python -m genie_cicd validate --sql          # the file is valid and every SQL runs
python -m genie_cicd plan   --env dev        # what would be created
python -m genie_cicd deploy --env dev        # creates "FreshCart Sales & Stock Assistant [DEV]"
python -m genie_cicd pull   --env dev        # round trip: commit what Databricks returns (authoritative format)
git commit -am "Genie: adopt the space exactly as Databricks exports it" && git push   # PR -> merge -> genie-release
```
Already have a space built by hand in dev? Skip the first deploy: set `GENIE_DEV_SPACE_ID` (or rely on the title)
and start with `pull --env dev`. Its export becomes version 1 in git.

> **Why the round trip matters.** The bootstrap file uses field names from Databricks' published bundle examples and
> CLI tests. A few optional fields (metric-view data sources, entity-matching flags, synonyms on SQL expressions)
> follow Databricks solution guidance. Deploying once and pulling back gives you the exact format your workspace
> uses, and that becomes the reviewed baseline.

### 8.7 · Using it day to day on Databricks
```bash
python -m genie_cicd snapshot --env dev                       # what the hourly job does
python -m genie_cicd history  --env prod                      # audit trail
python -m genie_cicd plan     --env prod                      # any drift in prod?
python -m genie_cicd rollback --env prod --to previous        # emergency (or use the genie-rollback workflow)
python -m genie_cicd rollback --env dev  --to 20261001T0921   # undo in dev to a snapshot (unique prefix of the id)
```

---
# 9 · Alternative: Declarative Automation Bundles

Since mid-2026 bundles have a native **`genie_spaces`** resource (direct deployment engine, the default since CLI
v1.14). The bundle commands map onto the same ideas:

| This repository | Bundle equivalent |
|---|---|
| `pull --env dev` | `databricks bundle generate genie-space --resource freshcart_assistant --force` (`--watch` keeps polling) |
| `deploy --env qa` | `databricks bundle deploy -t qa` |
| drift guard | the bundle tracks the etag and asks you to acknowledge remote changes before overwriting them |
| `{{catalog}}` render | not built in: the file is inlined as-is, so render per target first (`python -m genie_cicd render --env qa --out ...`) |
| history, backups, rollback, benchmark gate, smoke test | not provided by bundles: keep using `genie_cicd` for these |

```yaml
# databricks.yml (sketch)
bundle:
  name: freshcart-genie
  engine: direct
variables:
  warehouse_id: {description: SQL warehouse for the space}
resources:
  genie_spaces:
    freshcart_assistant:
      title: FreshCart Sales & Stock Assistant
      warehouse_id: ${var.warehouse_id}
      file_path: ./build/freshcart_assistant.geniespace.json     # rendered for the target by CI
      permissions:
        - {level: CAN_RUN, group_name: freshcart-business-users}
targets:
  dev:  {mode: development, workspace: {host: https://<dev-workspace>}}
  qa:   {workspace: {host: https://<qa-workspace>}}
  prod: {mode: production, workspace: {host: https://<prod-workspace>}}
```

**Choose one owner per space.** If a bundle deploys the space, replace the `deploy` steps in genie-release with
`render` + `databricks bundle deploy -t <env>`, and keep `validate`, `gate`, `smoke`, `snapshot` and `rollback` (which
then redeploys an older file through the bundle). Don't let both deploy the same space. A bonus of the bundle route:
it can also manage the space's **permissions**.

---
# 10 · Runbook

### Making a change
1. Edit the **dev** space in the UI (or edit the file and `deploy --env dev`).
2. Wait for the hourly dev-sync PR, or run `pull --env dev` on a branch and open a PR yourself.
3. Read the change list in the PR; get a review; genie-ci must be green.
4. Merge. genie-release deploys to QA and runs the benchmarks.
5. Approve the prod deployment in GitHub when QA is green. Done: the release is tagged.

### Adding a benchmark (recommended for every bug)
When Genie answers something wrongly, add the question with its correct SQL as a benchmark in dev, then follow
"Making a change". From then on, every release proves that bug stays fixed.

### Emergency rollback
GitHub → Actions → **genie-rollback** → environment `prod`, to `previous`, reason → approve. Or on a laptop:
`python -m genie_cicd rollback --env prod --to previous`. Then fix forward with a normal PR.

### "Deploy refused: drift"
Someone edited QA/prod in the UI. Look at the plan output. Good change → `pull --env prod` on a branch, PR. Unwanted
→ rerun genie-release with *allow drift* (the edit is kept in a backup record).

### "Objects missing in catalog ..."
The data layer of that environment is behind. Deploy the gold tables / metric views there first, then rerun.

### "Benchmark gate failed"
Open the job summary: it lists each `BAD` question with the expected and actual result. Fix in dev (often a dev
rollback to the last good snapshot), PR, release.

### What is **not** versioned by the space file
Conversations and feedback (they belong to users and stay in the space, which is never recreated), space
**permissions** (set once per environment, or by a bundle), the warehouse itself, and the data (versioned by its own
pipeline). Keep `deploy/genie/environments.yml` in git: it is versioned with everything else.

---
# 11 · Summary

* **The space is code:** one environment-neutral, canonical JSON file in git is the source of truth for every environment.
* **Every change has a version:** git for reviewed changes, hourly dev snapshots for UI edits, a backup before every deploy.
* **Promotion is one artifact moving forward:** dev → (PR) → main → QA (benchmarks) → approval → prod (smoke) → tag.
* **Mistakes are caught or reverted:** static + SQL checks on PRs, benchmark gate in QA, drift guard and etags on writes,
  automatic rollback on a failed prod smoke test, one-command rollback to any version.
* **It runs today:** `genie_cicd` uses the public Genie APIs through the Databricks SDK; the same code was exercised
  above against simulated workspaces and is covered by `tests/test_genie_cicd.py`.

| Term | Meaning |
|---|---|
| Serialized space | The JSON export of a Genie space; what `get_space(include_serialized_space=True)` returns and `create/update_space` accept |
| Environment-neutral | Catalog written as `{{catalog}}`, filled in per environment by `render` |
| Canonical form | Sorted, one-value-per-line JSON, so diffs show only real changes |
| Content hash | 12-character fingerprint of a space's content |
| Snapshot / backup / release | History records: periodic state / state before a deploy / what the pipeline deployed |
| Drift | Live space differs from the last release: edited outside the pipeline |
| Etag | Version stamp that makes concurrent overwrites fail instead of silently winning |
| Gate | Automatic pass/fail check a release must pass before moving on (benchmarks in QA) |
| Artifact promotion | Build once; deploy the same file to each environment |
| Rollback | Redeploy an earlier version through the same safe path |

In [29]:
shutil.rmtree(LAB, ignore_errors=True)        # clean up the lab
print("lab removed")

lab removed
